# 00 — Exploring Radiomics Data

**Objective:** Understand radiomics features before building ML models

**What you'll learn:**
1. What radiomics features look like in a CSV
2. Feature categories (first-order, shape, texture)
3. Exploratory Data Analysis (EDA) techniques
4. How to spot potential issues before modeling

**Time:** ~45 minutes

**Prerequisites:** Basic Python and pandas knowledge

## 1. Setup

In [ ]:
%pip install -q pandas numpy matplotlib seaborn scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configure plots
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('ggplot')
sns.set_palette('husl')

print('✓ Libraries imported')

## 2. Load a Radiomics Dataset

We'll load a pre-extracted radiomics dataset. You have two options:
- **Option A:** Download a real dataset from radMLBench (recommended)
- **Option B:** Use the synthetic dataset included in this project

In [ ]:
# Option A: Download real pre-extracted features from radMLBench
# Change DATASET_NAME to try different ones:
#   'LNDb'       - Lung nodule (173 samples, 105 features)
#   'HNSCC'      - Head-neck cancer (93 samples, 105 features)
#   'Ahn2021'    - Breast cancer MRI (114 samples, 108 features)

DATASET_NAME = 'LNDb'

try:
    url = f'https://github.com/aydindemircioglu/radMLBench/raw/refs/heads/main/datasets/{DATASET_NAME}.gz'
    df = pd.read_csv(url, compression='gzip')
    print(f'✓ Loaded real dataset: {DATASET_NAME}')
except Exception as e:
    print(f'⚠ Could not download {DATASET_NAME}: {e}')
    print('Falling back to synthetic data...')
    
    # Option B: Generate synthetic data
    np.random.seed(42)
    n = 100
    feature_defs = {
        'original_firstorder_Mean': (100, 20, 150, 30),
        'original_firstorder_Entropy': (3.5, 0.5, 4.5, 0.7),
        'original_firstorder_Energy': (10000, 2000, 20000, 4000),
        'original_firstorder_Variance': (500, 100, 800, 200),
        'original_firstorder_Skewness': (0.2, 0.3, 0.8, 0.4),
        'original_shape_VoxelVolume': (1000, 200, 2000, 400),
        'original_shape_Sphericity': (0.8, 0.1, 0.6, 0.15),
        'original_shape_SurfaceArea': (500, 80, 800, 150),
        'original_shape_Elongation': (0.7, 0.1, 0.55, 0.12),
        'original_shape_Flatness': (0.6, 0.08, 0.45, 0.1),
        'original_glcm_Contrast': (50, 10, 100, 20),
        'original_glcm_Correlation': (0.7, 0.1, 0.5, 0.15),
        'original_glcm_Homogeneity': (0.3, 0.05, 0.2, 0.04),
        'original_glcm_JointEntropy': (6.0, 0.5, 7.5, 0.8),
        'original_glrlm_GrayLevelNonUniformity': (100, 20, 200, 40),
        'original_glrlm_ShortRunEmphasis': (0.85, 0.05, 0.75, 0.08),
        'original_glszm_SizeZoneNonUniformity': (200, 40, 400, 80),
        'original_glszm_ZoneEntropy': (5.0, 0.6, 6.5, 0.8),
    }
    rows = []
    for i in range(n):
        is_mal = i >= n // 2
        row = {'ID': f'Patient_{i+1:03d}', 'Target': int(is_mal)}
        for feat, (bm, bs, mm, ms) in feature_defs.items():
            row[feat] = np.random.normal(mm if is_mal else bm, ms if is_mal else bs)
        rows.append(row)
    df = pd.DataFrame(rows)
    DATASET_NAME = 'Synthetic'
    print('✓ Using synthetic dataset')

print(f'\nDataset shape: {df.shape[0]} samples × {df.shape[1]} columns')

## 3. First Look at the Data

Always start by understanding what you're working with!

In [ ]:
# Preview first few rows
print('=== First 5 Rows ===')
df.head()

In [ ]:
# remove leading `1_` prefix from all column names that have it
rename_map = {c: c[2:] for c in df.columns if c.startswith('1_')}
if rename_map:
    df.rename(columns=rename_map, inplace=True)
    print(f"Renamed {len(rename_map)} columns:")
    for old, new in list(rename_map.items())[:10]:
        print(f"  {old} -> {new}")
else:
    print("No columns start with '1_'")

In [ ]:
# Basic info
print('=== Dataset Info ===')
print(f'Total samples:  {df.shape[0]}')
print(f'Total columns:  {df.shape[1]}')

# Identify column types
id_col = [c for c in df.columns if c.lower() in ['id', 'patient_id']][0] if any(c.lower() in ['id', 'patient_id'] for c in df.columns) else None
target_col = [c for c in df.columns if c.lower() in ['target', 'label']][0] if any(c.lower() in ['target', 'label'] for c in df.columns) else None
feature_cols = [c for c in df.columns if c not in [id_col, target_col]]

print(f'\nID column:      {id_col}')
print(f'Target column:  {target_col}')
print(f'Feature columns: {len(feature_cols)}')

# Class distribution
if target_col:
    print(f'\n=== Class Distribution ===')
    class_counts = df[target_col].value_counts()
    for cls, count in class_counts.items():
        pct = count / len(df) * 100
        print(f'  Class {cls}: {count} samples ({pct:.1f}%)')
    
    ratio = class_counts.min() / class_counts.max()
    if ratio < 0.5:
        print(f'\n  ⚠ Dataset is IMBALANCED (minority ratio: {ratio:.2f})')
        print('  → Consider: stratified sampling, class weights, or SMOTE')
    else:
        print(f'\n  ✓ Dataset is reasonably balanced (ratio: {ratio:.2f})')

## 4. Understanding Feature Categories

Radiomics features follow a naming convention: `original_{category}_{name}`

| Category | Prefix | What It Measures |
|----------|--------|------------------|
| First-order | `firstorder` | Intensity distribution (mean, variance, entropy...) |
| Shape | `shape` | Tumor geometry (volume, sphericity, elongation...) |
| GLCM | `glcm` | Spatial intensity patterns (contrast, correlation...) |
| GLRLM | `glrlm` | Run-length patterns (long runs, short runs...) |
| GLSZM | `glszm` | Size-zone patterns (large areas, small areas...) |
| GLDM | `gldm` | Dependency patterns |
| NGTDM | `ngtdm` | Neighborhood differences |

In [ ]:
# Count features by category
categories = {}
for col in feature_cols:
    parts = col.split('_')
    if len(parts) >= 3:
        # e.g., 'original_firstorder_Mean' → 'firstorder'
        cat = parts[1] if parts[0] == 'original' else parts[0]
    else:
        cat = 'other'
    categories.setdefault(cat, []).append(col)

print('=== Features by Category ===')
cat_data = []
for cat, feats in sorted(categories.items(), key=lambda x: -len(x[1])):
    print(f'  {cat:20s}: {len(feats):3d} features')
    cat_data.append({'Category': cat, 'Count': len(feats)})

# Visualize
cat_df = pd.DataFrame(cat_data)
fig, ax = plt.subplots(figsize=(8, 4))
colors = sns.color_palette('husl', len(cat_df))
ax.barh(cat_df['Category'], cat_df['Count'], color=colors)
ax.set_xlabel('Number of Features')
ax.set_title('Feature Categories in This Dataset')
for i, v in enumerate(cat_df['Count']):
    ax.text(v + 0.5, i, str(v), va='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Data Quality Checks

Before any analysis, check for common data issues.

In [ ]:
# Check for missing values
missing = df[feature_cols].isnull().sum()
n_missing = (missing > 0).sum()
print(f'=== Missing Values ===')
if n_missing == 0:
    print('  ✓ No missing values found')
else:
    print(f'  ⚠ {n_missing} features have missing values:')
    print(missing[missing > 0].sort_values(ascending=False).head(10))

# Check for infinite values
inf_count = np.isinf(df[feature_cols].select_dtypes(include=[np.number])).sum().sum()
print(f'\n=== Infinite Values ===')
print(f'  {"✓ None found" if inf_count == 0 else f"⚠ {inf_count} infinite values detected"}')

# Check for constant features (zero variance)
variances = df[feature_cols].var()
constant_feats = variances[variances == 0]
print(f'\n=== Constant Features ===')
if len(constant_feats) == 0:
    print('  ✓ No constant features')
else:
    print(f'  ⚠ {len(constant_feats)} constant features (should be removed):')
    for f in constant_feats.index:
        print(f'    - {f}')

# Summary statistics
print(f'\n=== Summary Statistics (first 5 features) ===')
df[feature_cols[:5]].describe().round(2)

## 6. Feature Distributions

Visualize how features differ between classes — this gives intuition about which features might be useful for classification.

In [ ]:
# Select interesting features to visualize
# Try to pick one from each category
viz_features = []
for cat in ['firstorder', 'shape', 'glcm', 'glrlm', 'glszm']:
    matching = [f for f in feature_cols if cat in f.lower()]
    if matching:
        viz_features.append(matching[0])

# Add more if we have fewer than 6
for f in feature_cols:
    if f not in viz_features and len(viz_features) < 6:
        viz_features.append(f)

# Plot distributions by class
n_plots = min(6, len(viz_features))
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, feature in enumerate(viz_features[:n_plots]):
    ax = axes[idx]
    for cls in sorted(df[target_col].unique()):
        data = df[df[target_col] == cls][feature].dropna()
        label = f'Class {cls} (n={len(data)})'
        ax.hist(data, alpha=0.5, bins=20, label=label, density=True)
    
    # Shorten long names
    short_name = feature.replace('original_', '').replace('_', ' ')
    ax.set_title(short_name, fontsize=10)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Class', fontsize=14, fontweight='bold')
plt.tight_layout()
# plt.savefig('../outputs/00_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Look for features where the two distributions are well-separated!')

## 7. Correlation Analysis

Radiomics features are often highly correlated. Understanding this helps with feature selection.

In [ ]:
# Compute correlation matrix
corr = df[feature_cols].corr()

# Count highly correlated pairs
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr_pairs = []
for col in upper.columns:
    for row in upper.index:
        if abs(upper.loc[row, col]) > 0.9:
            high_corr_pairs.append((row, col, upper.loc[row, col]))

print(f'=== Correlation Analysis ===')
print(f'Total feature pairs: {len(feature_cols) * (len(feature_cols)-1) // 2}')
print(f'Highly correlated pairs (|r| > 0.9): {len(high_corr_pairs)}')
if high_corr_pairs:
    print(f'\nTop 5 most correlated pairs:')
    for f1, f2, r in sorted(high_corr_pairs, key=lambda x: abs(x[2]), reverse=True)[:5]:
        print(f'  {f1[:30]:30s} ↔ {f2[:30]:30s}  r={r:.3f}')

# Heatmap (subset of features for readability)
n_show = min(20, len(feature_cols))
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    corr.iloc[:n_show, :n_show], 
    annot=n_show <= 15, 
    fmt='.1f', 
    cmap='RdBu_r', 
    center=0, 
    ax=ax,
    xticklabels=[f.replace('original_', '')[:20] for f in feature_cols[:n_show]],
    yticklabels=[f.replace('original_', '')[:20] for f in feature_cols[:n_show]]
)
ax.set_title(f'Feature Correlation Matrix (first {n_show} features)', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
# plt.savefig('../outputs/00_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Feature Scale Comparison

Radiomics features have very different scales. This is why **standardization** is critical before ML.

In [ ]:
# Show the scale problem
sample_feats = feature_cols[:min(10, len(feature_cols))]
ranges = df[sample_feats].apply(lambda x: x.max() - x.min())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before standardization
axes[0].boxplot([df[f].dropna().values for f in sample_feats], labels=range(1, len(sample_feats)+1))
axes[0].set_title('Before Standardization', fontsize=12)
axes[0].set_ylabel('Raw Value')
axes[0].set_xlabel('Feature Index')

# After standardization
from sklearn.preprocessing import StandardScaler
scaled = StandardScaler().fit_transform(df[sample_feats].fillna(0))
axes[1].boxplot([scaled[:, i] for i in range(scaled.shape[1])], labels=range(1, len(sample_feats)+1))
axes[1].set_title('After Standardization (z-score)', fontsize=12)
axes[1].set_ylabel('Standardized Value')
axes[1].set_xlabel('Feature Index')

plt.suptitle('Why Standardization Matters', fontsize=14, fontweight='bold')
plt.tight_layout()
# plt.savefig('../outputs/00_standardization_demo.png', dpi=150, bbox_inches='tight')
plt.show()

print('📏 Notice how raw features have very different scales!')
print('   After standardization, all features have mean≈0 and std≈1.')
print('   This is essential for SVM and Logistic Regression to work properly.')

## 9. Quick Univariate Analysis

Which individual features best separate the two classes? This gives us a preview of what feature selection will reveal.

In [ ]:
# Statistical test: which features differ significantly between classes?
from scipy import stats

results = []
for feat in feature_cols:
    class0 = df[df[target_col] == 0][feat].dropna()
    class1 = df[df[target_col] == 1][feat].dropna()
    if len(class0) > 1 and len(class1) > 1:
        stat, pval = stats.mannwhitneyu(class0, class1, alternative='two-sided')
        # Effect size (rank-biserial correlation)
        n1, n2 = len(class0), len(class1)
        effect = 1 - (2 * stat) / (n1 * n2)
        results.append({'Feature': feat, 'p-value': pval, 'Effect Size': abs(effect)})

results_df = pd.DataFrame(results).sort_values('p-value')

# Show top discriminative features
print('=== Top 10 Most Discriminative Features (Mann-Whitney U test) ===')
print(f'{"Feature":50s} {"p-value":>12s} {"Effect Size":>12s}')
print('-' * 76)
for _, row in results_df.head(10).iterrows():
    sig = '***' if row['p-value'] < 0.001 else '**' if row['p-value'] < 0.01 else '*' if row['p-value'] < 0.05 else ''
    feat_short = row['Feature'].replace('original_', '')[:48]
    print(f'{feat_short:50s} {row["p-value"]:>10.2e} {sig:>2s} {row["Effect Size"]:>10.3f}')

# Visualize top features
top_n = min(10, len(results_df))
fig, ax = plt.subplots(figsize=(10, 5))
top = results_df.head(top_n)
colors = ['#2E86AB' if p < 0.05 else '#cccccc' for p in top['p-value']]
ax.barh(range(top_n), top['Effect Size'].values, color=colors)
ax.set_yticks(range(top_n))
ax.set_yticklabels([f.replace('original_', '')[:35] for f in top['Feature']], fontsize=9)
ax.set_xlabel('Effect Size (rank-biserial correlation)')
ax.set_title('Top Discriminative Features', fontsize=13)
ax.invert_yaxis()
plt.tight_layout()
# plt.savefig('../outputs/00_top_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Key Takeaways

### ✅ Before you proceed to Machine Learning, make sure you understand:

1. **Dataset size**: How many samples and features do you have?
2. **Class balance**: Is it balanced? If not, what strategy will you use?
3. **Feature categories**: What types of features are present? (shape, texture, etc.)
4. **Data quality**: Any missing values, infinities, or constant features?
5. **Feature correlations**: Are many features highly correlated? (common in radiomics!)
6. **Scale differences**: Why is standardization essential?
7. **Discriminative features**: Which features look most promising?

### Next Steps

→ Continue to **`01_feature_extraction.ipynb`** to learn how to extract features from images

→ Or skip to **`02_machine_learning.ipynb`** to build and evaluate ML models